# M1 skeleton, tau2 part, on a Colab L4

Adds the 3 **tau2-bench-verified airline** tasks to the M1 skeleton: **4 baseline configs × 3 tasks × 2 seeds + R1/R2/R3 × 3 tasks × 1 seed = 33 episodes**, with the simulated user and the NL judge self-hosted (gpt-oss-20b, greedy) behind the C4 cache. The 220 BFCL episodes finished on Oct 24–25 carry over unchanged (same run_ids), so `report.md` covers all 253.

* Runtime: **L4** (Runtime → Change runtime type → L4, High-RAM). 1.7 compute units/h against 5.4 for the A100; 22.5 GB holds the 0.6B agent and the 12 GB simulator together.
* The CPU-backend configuration's agent runs on this runtime's CPU cores; the simulator stays on the GPU.
* Resumable: re-run all cells after a disconnect. The C4 cache keeps every simulator reply, so a resumed episode does not pay for them again.
* Development data (M1), excluded from the study.


In [ ]:
# 1. Drive + paths
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive'
PILOT_MODELS = f'{DRIVE}/CMPE295A_pilot_2026-10-08/models'
CACHE = f'{DRIVE}/CMPE295A_m1_cache'
OUT = f'{DRIVE}/CMPE295A_m1_runs'
OLD = f'{OUT}/skeleton/d92a85e486'      # the BFCL-only skeleton directory (220 finished episodes)
BRANCH = 'm1/skeleton'
os.environ.update(TESTBED_OUT=OUT, TESTBED_MODELS='/content/models', TESTBED_CACHE_LOCAL='/content/simuser_cache.sqlite',
                  LLAMA_CUDA_BIN='/content/cmpe_295a/llama.cpp/build-cuda/bin/llama-server',
                  LLAMA_CPU_BIN='/content/cmpe_295a/llama.cpp/build-cpu/bin/llama-server',
                  LITELLM_LOCAL_MODEL_COST_MAP='True')
THREADS = os.cpu_count()
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2

In [ ]:
# 2. Code
import subprocess
if not os.path.exists('/content/cmpe_295a'):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/VritikaMalhotra/cmpe_295a', '/content/cmpe_295a'], check=True)
else:
    subprocess.run(['git', '-C', '/content/cmpe_295a', 'pull', '--ff-only'], check=True)
%cd /content/cmpe_295a
!git log -1 --oneline

In [ ]:
# 3. Environment: testbed + llama.cpp (restored from the Drive cache) + tau2-bench-verified at the pinned commit
!bash scripts/setup_env.sh --llama --cuda --tau2 --cache {CACHE}

In [ ]:
# 4. Model files: the agent GGUFs (same files as the BFCL run, SHA-256 checked) + the simulated user (12 GB, cached on Drive)
!mkdir -p /content/models
!cp -n {PILOT_MODELS}/Qwen3-0.6B-BF16.gguf {PILOT_MODELS}/Qwen3-0.6B-Q4_K_M.gguf {PILOT_MODELS}/Qwen3-0.6B-Q4_0-pure.gguf /content/models/
!cp -n {CACHE}/Qwen3-0.6B-Q8_0.gguf {CACHE}/r3/Qwen3-0.6B-S7QAD-Q4_0-pure.gguf /content/models/
!bash scripts/get_simuser_model.sh /content/models {CACHE}/simuser

In [ ]:
# 5. Dry run; carries the 220 finished BFCL episodes over from the BFCL-only directory (same run_ids)
!.venv/bin/python -m testbed.run +experiment=skeleton dry_run=true import_from={OLD}

In [ ]:
# 6. The 33 tau2 episodes (re-run after a disconnect; it resumes)
!.venv/bin/python -m testbed.run +experiment=skeleton local=true only_benchmark=tau2_verified_airline cpu_threads={THREADS} 2>&1 | tee -a {OUT}/colab_tau2_run.log | tail -45

In [ ]:
# 7. FR-E6 check: replay one finished tau2 episode from the cache alone (no simulator process) -> zero misses
import glob, json
d = sorted(glob.glob(f'{OUT}/skeleton/*/specs.jsonl'), key=os.path.getmtime)[-1].rsplit('/', 1)[0]
specs = [json.loads(l) for l in open(f'{d}/specs.jsonl')]
done = [s['run_id'] for s in specs if s['benchmark'].startswith('tau2') and s['backend'] == 'cuda'
        and os.path.exists(f"{d}/episodes/{s['run_id']}.json")]
print(d, len(done), 'finished cuda tau2 episodes')
if done:
    !.venv/bin/python -m testbed.run +experiment=skeleton replay={done[0]} cpu_threads={THREADS} 2>&1 | tail -3

In [ ]:
# 8. Report (all 253 episodes) and the C4 ledger
!.venv/bin/python -m testbed.run +experiment=skeleton report_only=true report_partial=true 2>&1 | tail -1
!.venv/bin/python -m testbed.cache report {d}/simuser_cache.sqlite | head -40
from IPython.display import Markdown, display
display(Markdown(open(f'{d}/report.md').read()))

In [ ]:
# 9. Release this runtime (results are on Drive). Set RELEASE = False to keep it.
RELEASE = True
if RELEASE:
    from google.colab import runtime
    runtime.unassign()